# EDA курсора

Абляция показала, что группа `cursor` — самая сильная (без неё CV P@R падает на ~15 п.п.),
хотя в ней всего 5 простых признаков. Цель ноутбука — понять, **чем именно** курсор ботов
отличается от курсора людей, чтобы сделать признаки точнее.

Курсор есть только на web, поэтому весь анализ — по web-кукам train.
События берём через `clean_events` из `features.py` (окно, без дублей, отсортированы) —
ровно те же, что видит модель.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from features import load_data, clean_events

sns.set_theme(style='whitegrid')
SCREEN_W, SCREEN_H = 1920, 1080

train, test, events = load_data()
ev = clean_events(events, train).merge(train[['cookie_id', 'target']], on='cookie_id')
web = ev[ev.plat == 'web'].copy()
web['has_ptr'] = web.pointer_x.notna()
print('web-событий:', len(web), ' web-куки:', web.cookie_id.nunique())

## 1. Есть ли курсор вообще

Для каждой web-куки считаем долю её событий с координатами (`pointer_share`).
Смотрим распределение этой доли по классам и по семействам клиента.

In [ ]:
ck = web.groupby('cookie_id').agg(target=('target', 'first'), ua_fam=('ua_fam', 'first'),
                                  n=('has_ptr', 'size'), pointer_share=('has_ptr', 'mean'))
ck['ptr_cat'] = pd.cut(ck.pointer_share, [-0.01, 0, 0.99, 1], labels=['нет совсем', 'частично', 'всегда'])

print('доля куки внутри класса:')
print(pd.crosstab(ck.ptr_cat, ck.target, normalize='columns').round(3))
print()
print('число куки по семействам клиента:')
pd.crosstab([ck.ua_fam, ck.ptr_cat], ck.target)

In [ ]:
# есть ли курсор у разных типов событий
web.groupby('event_name').has_ptr.mean().round(2).sort_values()

## 2. Где стоит курсор

Тепловая карта экрана 1920×1080: какая доля всех координат класса попала в каждую клетку.
Если курсор ставится случайно по всему экрану, карта будет равномерной.

In [ ]:
ptr = web[web.has_ptr].copy()

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for i, (y, name) in enumerate([(0, 'люди'), (1, 'боты')]):
    q = ptr[ptr.target == y]
    H, _, _ = np.histogram2d(q.pointer_y, q.pointer_x, bins=[12, 16], range=[[0, SCREEN_H], [0, SCREEN_W]])
    sns.heatmap(H / H.sum() * 100, ax=ax[i], cmap='viridis', cbar_kws={'label': '% координат'},
                xticklabels=False, yticklabels=False)
    ax[i].set_title(f'{name}: где курсор ({len(q)} событий)')
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(data=ptr, x='pointer_x', hue='target', bins=48, stat='density', common_norm=False, ax=ax[0])
ax[0].axvline(SCREEN_W / 2, ls='--', c='gray'); ax[0].set_title('координата X')
sns.histplot(data=ptr, x='pointer_y', hue='target', bins=36, stat='density', common_norm=False, ax=ax[1])
ax[1].axvline(SCREEN_H / 2, ls='--', c='gray'); ax[1].set_title('координата Y')
plt.tight_layout(); plt.show()

# для сравнения: у равномерного распределения на [0, W] std = W / sqrt(12)
print('std равномерного:  X', round(SCREEN_W / np.sqrt(12)), ' Y', round(SCREEN_H / np.sqrt(12)))
print(ptr.groupby('target')[['pointer_x', 'pointer_y']].std().round(0))

## 3. Курсор на уровне куки

Для каждой куки с ≥ 5 координатами: центр масс (средние x, y) и разброс (std x, std y).
Слева — где находится «облако» курсора куки, справа — насколько оно широкое.

In [ ]:
pc = ptr.groupby('cookie_id').agg(target=('target', 'first'), n_ptr=('pointer_x', 'size'),
                                  mx=('pointer_x', 'mean'), my=('pointer_y', 'mean'),
                                  sx=('pointer_x', 'std'), sy=('pointer_y', 'std'))
pc5 = pc[pc.n_ptr >= 5]
print(pc5.target.value_counts().rename('куки с >=5 координатами'))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for y, c, a in [(0, 'tab:blue', 0.15), (1, 'tab:orange', 0.6)]:
    q = pc5[pc5.target == y]
    ax[0].scatter(q.mx, q.my, s=8, c=c, alpha=a, label=f'target={y}')
    ax[1].scatter(q.sx, q.sy, s=8, c=c, alpha=a, label=f'target={y}')
ax[0].set(xlim=(0, SCREEN_W), ylim=(SCREEN_H, 0), title='центр масс курсора куки', xlabel='mean x', ylabel='mean y')
ax[1].axvline(SCREEN_W / np.sqrt(12), ls='--', c='gray'); ax[1].axhline(SCREEN_H / np.sqrt(12), ls='--', c='gray')
ax[1].set(title='разброс курсора куки (пунктир — равномерный)', xlabel='std x', ylabel='std y')
ax[0].legend(); ax[1].legend()
plt.tight_layout(); plt.show()

## 4. Движение между событиями

Для соседних событий куки, у которых оба раза есть курсор: на сколько пикселей он сдвинулся
и с какой скоростью (пиксели / секунды между событиями).

In [ ]:
g = ptr.groupby('cookie_id')
ptr['step'] = np.hypot(g.pointer_x.diff(), g.pointer_y.diff())
ptr['step_dt'] = g.event_ts.diff().dt.total_seconds()
ptr['speed'] = ptr.step / ptr.step_dt.replace(0, np.nan)

print(ptr.groupby('target').step.describe().round(0))
print()
print('доля шагов без движения (step == 0):')
print(ptr.groupby('target').step.apply(lambda s: (s == 0).mean()).round(4))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(data=ptr, x='step', hue='target', bins=50, stat='density', common_norm=False, ax=ax[0])
ax[0].set_title('сдвиг курсора между соседними событиями, px')
sns.histplot(data=ptr[ptr.speed > 0], x='speed', hue='target', log_scale=True, bins=50,
             stat='density', common_norm=False, ax=ax[1])
ax[1].set_title('скорость курсора, px/с')
plt.tight_layout(); plt.show()

## 5. Курсор и тип события

Медианное положение курсора по типам событий. Если бот кликает по кнопкам «по координатам»,
у контактных событий положение может быть почти фиксированным.

In [ ]:
med = ptr.groupby(['event_name', 'target'])[['pointer_x', 'pointer_y']].median().unstack('target').round(0)
med.columns = [f'{c}_{"bot" if t else "human"}' for c, t in med.columns]
med

In [ ]:
# «круглые» координаты: кратные 10 по обеим осям
ptr['round10'] = (ptr.pointer_x % 10 == 0) & (ptr.pointer_y % 10 == 0)
print('доля координат, кратных 10 по обеим осям (ожидаемо при случайности: 1%):')
print(ptr.groupby('target').round10.mean().round(4))